# Phase 6 — Interactive FitInsight evidence explorer

## Research objective
Explore validated activity, clustering, PCA, DBSCAN, association-rule, anomaly-screening, and sleep/time evidence without refitting models or changing the analytical results. All plots in this notebook are generated from the stored evidence tables.

## Study and method
The source is a historical 2016 wearable convenience sample. Activity is recorded at participant-day grain; clustering, PCA, and DBSCAN summaries are participant-level; sleep episodes remain a separate grain. Widget changes filter these existing results only. Cluster IDs are arbitrary, association rules are descriptive, and anomaly flags are review candidates—not clinical findings or confirmed errors.

In [1]:
from pathlib import Path
import csv, json, html, math
import ipywidgets as widgets
from IPython.display import display, HTML, SVG

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'processed').is_dir() and (candidate / 'reports' / 'generated').is_dir():
            return candidate
    raise RuntimeError('Open this notebook from the FitInsight project or one of its folders.')

ROOT = find_project_root()
def read_csv(path):
    with Path(path).open(encoding='utf-8-sig', newline='') as stream:
        return list(csv.DictReader(stream))

def bool_value(value):
    if isinstance(value, bool): return value
    if value is None: return False
    return str(value).strip().casefold() in {'true', '1', 'yes'}

def json_list(value):
    if isinstance(value, list): return value
    if value is None or value == '': return []
    parsed = json.loads(value) if isinstance(value, str) else value
    return parsed if isinstance(parsed, list) else []

def number(row, key, default=0.0):
    value = row.get(key, '')
    return default if value in ('', None) else float(value)

def table_html(rows, columns, limit=15):
    rows = list(rows)
    if not rows: return '<p class="empty-state">No records match these filters.</p>'
    header = ''.join(f'<th>{html.escape(str(name))}</th>' for name in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{html.escape(str(row.get(name, "")))}</td>' for name in columns) + '</tr>' for row in rows[:limit])
    return '<div style="overflow-x:auto"><table style="border-collapse:collapse;width:100%"><thead><tr>' + header + '</tr></thead><tbody>' + body + '</tbody></table></div>'

PALETTE = ['#277da1', '#f8961e', '#43aa8b', '#9b5de5', '#f94144', '#577590']
def color_for(label):
    try: return PALETTE[int(label) % len(PALETTE)]
    except (ValueError, TypeError): return PALETTE[0]

activity = read_csv(ROOT / 'data/processed/daily_activity_clean.csv')
clusters = read_csv(ROOT / 'reports/generated/activity_clustering/tables/participant_clusters.csv')
pca_coordinates = read_csv(ROOT / 'reports/generated/activity_clustering/tables/pca_coordinates.csv')
pca_loadings = read_csv(ROOT / 'reports/generated/activity_clustering/tables/pca_loadings.csv')
pca_variance = read_csv(ROOT / 'reports/generated/activity_clustering/tables/pca_variance.csv')
dbscan_scores = read_csv(ROOT / 'reports/generated/activity_clustering/tables/dbscan_scores.csv')
kmeans_evidence = json.loads((ROOT / 'reports/evidence/P4/kmeans_evaluation.json').read_text(encoding='utf-8'))
rules = read_csv(ROOT / 'reports/generated/associations/association_rules.csv')
candidates = read_csv(ROOT / 'reports/generated/anomalies/candidate_registry.csv')
sleep = read_csv(ROOT / 'reports/generated/eda/tables/sleep_episodes.csv')
time_accounting = read_csv(ROOT / 'reports/generated/eda/tables/time_accounting_by_date.csv')
assert len(activity) == 457 and len({row['id'] for row in activity}) == 35
assert len([row for row in clusters if row.get('all_days_cluster', '').strip()]) == 35
assert len([row for row in clusters if row.get('complete_days_cluster', '').strip()]) == 24
assert sum(bool_value(row.get('any_candidate')) for row in candidates) == 217
assert len({row['activity_date'] for row in time_accounting}) == 23
panel_state = {}
print('Loaded validated records:', {'activity_days': len(activity), 'participants': 35, 'all_days_assignments': 35, 'complete_day_assignments': 24, 'rules': len(rules), 'flagged_days': 217, 'all_registry_days': len(candidates), 'time_accounting_dates': 23, 'sleep_episodes': len(sleep)})

Loaded validated records: {'activity_days': 457, 'participants': 35, 'all_days_assignments': 35, 'complete_day_assignments': 24, 'rules': 860, 'flagged_days': 217, 'all_registry_days': 457, 'time_accounting_dates': 23, 'sleep_episodes': 556}


## 6.1 Activity and recorded-day coverage
Select a participant and a recorded-day completeness group. The SVG is regenerated from the filtered activity rows: the x-axis is activity date, the y-axis is mean recorded steps, and its description reports both dates plotted and participant-day observations contributing. Completeness is the stored 1,440-recorded-minute flag, not proof of device wear.

In [2]:
participant = widgets.Dropdown(options=['All participants'] + sorted({row['id'] for row in activity}), description='Participant')
coverage = widgets.Dropdown(options=['All days', 'Complete only', 'Incomplete only'], description='Coverage')
out_activity = widgets.HTML()

def activity_svg(rows):
    by_date = {}
    for row in rows: by_date.setdefault(row['activity_date'], []).append(number(row, 'total_steps'))
    points = sorted((day, sum(values) / len(values)) for day, values in by_date.items())
    width, height, left, right, top, bottom = 760, 300, 70, 735, 35, 245
    if not points:
        return f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}"><text x="30" y="60">No rows match these filters.</text></svg>'
    low, high = min(value for _, value in points), max(value for _, value in points)
    span = max(high - low, 1)
    xy = [(left + i * (right-left) / max(len(points)-1, 1), bottom - (value-low) * (bottom-top) / span) for i, (_, value) in enumerate(points)]
    polyline = ' '.join(f'{x:.1f},{y:.1f}' for x, y in xy)
    desc = f'{len(points)} dates plotted; {len(rows)} participant-day records contribute; mean steps per date.'
    return f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}" role="img"><title>Mean recorded steps by activity date</title><desc>{html.escape(desc)}</desc><rect width="100%" height="100%" fill="white"/><line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#555"/><line x1="{left}" y1="{bottom}" x2="{right}" y2="{bottom}" stroke="#555"/><polyline fill="none" stroke="#277da1" stroke-width="2" points="{polyline}"/><text x="{(left+right)/2}" y="282" text-anchor="middle">Activity date</text><text x="18" y="150" transform="rotate(-90 18 150)" text-anchor="middle">Mean recorded steps (steps/day)</text><text x="{left}" y="24">{len(points)} dates · {len(rows)} participant-days · mean {sum(number(r,'total_steps') for r in rows)/len(rows):.0f} steps</text></svg>'

def render_activity(change=None):
    rows = [row for row in activity if (participant.value == 'All participants' or row['id'] == participant.value) and (coverage.value == 'All days' or bool_value(row['is_complete_day']) == (coverage.value == 'Complete only'))]
    panel_state['activity_rows'] = rows
    panel_state['activity_svg'] = activity_svg(rows)
    if rows:
        steps = [number(row, 'total_steps') for row in rows]
        summary = f'{len(rows)} participant-days · {len({r["id"] for r in rows})} participants · {len({r["activity_date"] for r in rows})} dates · mean {sum(steps)/len(steps):.1f} steps/day'
    else: summary = 'No participant-days match these filters.'
    out_activity.value = (f'<p>{summary}</p>' + panel_state['activity_svg'] +
                          table_html(rows, ['id', 'activity_date', 'total_steps', 'total_recorded_minutes', 'is_complete_day']))

participant.observe(render_activity, names='value')
coverage.observe(render_activity, names='value')
render_activity()

## 6.2 Participant-level K-Means assignments
The all-days model has 35 eligible participants and three labels; the complete-day model has 24 participants and two labels. Blank assignments are excluded and explicitly counted as ineligible for that cohort. Cluster labels are cohort-specific and arbitrary—not fitness categories.

In [3]:
cluster_cohort = widgets.Dropdown(options=['all_days', 'complete_days'], value='all_days', description='Cohort')
cluster_field = {'all_days': 'all_days_cluster', 'complete_days': 'complete_days_cluster'}
cluster_choice = widgets.Dropdown(options=['All clusters'], description='Cluster')
out_clusters = widgets.HTML()

def valid_cluster_rows(cohort):
    column = cluster_field[cohort]
    return [row for row in clusters if row.get(column, '').strip() != '']

def refresh_cluster_options(change=None):
    labels = sorted({row[cluster_field[cluster_cohort.value]] for row in valid_cluster_rows(cluster_cohort.value)}, key=int)
    options = ['All clusters'] + labels
    if tuple(cluster_choice.options) != tuple(options):
        cluster_choice.options = options
        if cluster_choice.value not in options: cluster_choice.value = 'All clusters'
    render_clusters()

def render_clusters(change=None):
    cohort = cluster_cohort.value
    field = cluster_field[cohort]
    eligible = valid_cluster_rows(cohort)
    selected = [row for row in eligible if cluster_choice.value == 'All clusters' or row[field] == cluster_choice.value]
    labels = sorted({row[field] for row in eligible}, key=int)
    panel_state['cluster_rows'] = selected
    panel_state['cluster_counts'] = {label: sum(row[field] == label for row in eligible) for label in labels}
    panel_state['cluster_eligible_count'] = len(eligible)
    panel_state['cluster_excluded_count'] = len(clusters) - len(eligible)
    summary = f'{cohort}: {len(eligible)} eligible participant assignments · {len(labels)} clusters · {len(clusters)-len(eligible)} rows excluded for blank cohort assignment.'
    out_clusters.value = f'<p>{summary}</p>' + table_html([{'id': row['id'], 'cluster': row[field], 'cohort': cohort} for row in selected], ['id', 'cohort', 'cluster'], 35)

cluster_cohort.observe(refresh_cluster_options, names='value')
cluster_choice.observe(render_clusters, names='value')
refresh_cluster_options()

## 6.3 PCA exploration
This panel filters the committed PCA coordinates by cohort and optional participant. PC1 and PC2 are projections of four standardized participant features; the stored explained-variance ratios and loadings are displayed alongside the plot. Changing controls does not recompute PCA.

In [4]:
pca_cohort = widgets.Dropdown(options=['all_days', 'complete_days'], value='all_days', description='Cohort')
pca_participant = widgets.Dropdown(options=['All participants'], description='Highlight person')
out_pca = widgets.HTML()

def refresh_pca_people(change=None):
    people = sorted({row['id'] for row in pca_coordinates if row['cohort'] == pca_cohort.value})
    options = ['All participants'] + people
    pca_participant.options = options
    if pca_participant.value not in options: pca_participant.value = 'All participants'
    render_pca()

def pca_svg(rows, cohort, selected_id, ratios):
    width, height, left, right, top, bottom = 780, 430, 80, 735, 45, 340
    if not rows: return '<svg xmlns="http://www.w3.org/2000/svg" width="780" height="430"><text x="24" y="50">No PCA coordinates match this cohort.</text></svg>'
    xs = [number(row, 'PC1') for row in rows]; ys = [number(row, 'PC2') for row in rows]
    xmin, xmax = min(xs), max(xs); ymin, ymax = min(ys), max(ys)
    if xmax == xmin: xmin -= 1; xmax += 1
    if ymax == ymin: ymin -= 1; ymax += 1
    points = []
    for row in rows:
        x = left + (number(row, 'PC1')-xmin)*(right-left)/(xmax-xmin)
        y = bottom - (number(row, 'PC2')-ymin)*(bottom-top)/(ymax-ymin)
        stroke = '#111827' if selected_id == row['id'] else 'white'
        radius = 8 if selected_id == row['id'] else 5
        points.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="{radius}" fill="{color_for(row["cluster"])}" stroke="{stroke}" stroke-width="2"><title>Participant {html.escape(row["id"])} · cluster {row["cluster"]}</title></circle>')
    legend = ''.join(f'<circle cx="{left+i*120}" cy="385" r="6" fill="{color_for(label)}"/><text x="{left+12+i*120}" y="389">Cluster {html.escape(label)}</text>' for i,label in enumerate(sorted({row['cluster'] for row in rows}, key=int)))
    desc = f'{len(rows)} participant coordinates; {cohort}; PC1 explains {ratios[0]:.1%}, PC2 explains {ratios[1]:.1%}.'
    return f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}" role="img"><title>PCA participant projection · {html.escape(cohort)}</title><desc>{html.escape(desc)}</desc><rect width="100%" height="100%" fill="white"/><line x1="{left}" y1="{bottom}" x2="{right}" y2="{bottom}" stroke="#475569"/><line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#475569"/><text x="{(left+right)/2}" y="365" text-anchor="middle">PC1 · {ratios[0]:.1%} explained variance</text><text x="22" y="195" transform="rotate(-90 22 195)" text-anchor="middle">PC2 · {ratios[1]:.1%} explained variance</text>{"".join(points)}{legend}<text x="90" y="25">{len(rows)} participants · stored coordinates · no refit</text></svg>'

def render_pca(change=None):
    cohort = pca_cohort.value
    rows = [row for row in pca_coordinates if row['cohort'] == cohort and (pca_participant.value == 'All participants' or row['id'] == pca_participant.value)]
    variances = sorted([row for row in pca_variance if row['cohort'] == cohort], key=lambda row: int(row['component'].replace('PC','')))
    ratios = [number(row, 'variance_ratio') for row in variances[:2]]
    panel_state['pca_rows'] = rows
    panel_state['pca_ratios'] = ratios
    panel_state['pca_svg'] = pca_svg(rows, cohort, pca_participant.value, ratios)
    summary = f'<p>{len(rows)} displayed participant(s); first two PCs explain {sum(ratios):.1%} cumulatively for the {html.escape(cohort)} cohort.</p>'
    loadings_html = table_html([row for row in pca_loadings if row['cohort'] == cohort], ['feature', 'PC1', 'PC2'], 10)
    out_pca.value = panel_state['pca_svg'] + summary + loadings_html

pca_cohort.observe(refresh_pca_people, names='value')
pca_participant.observe(render_pca, names='value')
refresh_pca_people()

## 6.4 DBSCAN parameter sensitivity
The controls select rows from the validated `eps` × `min_samples` grid. The chart and table report the stored number of clusters, noise points, and silhouette where defined; a small K-Means reference is shown for context. This panel does not create DBSCAN assignments or refit a model. Density outcomes are parameter-sensitive and noise is excluded from partition scores.

In [5]:
dbscan_min_samples = widgets.Dropdown(options=sorted({int(row['min_samples']) for row in dbscan_scores}), value=3, description='min_samples')
dbscan_eps = widgets.Dropdown(options=['All eps'] + sorted({row['eps'] for row in dbscan_scores}, key=float), value='All eps', description='eps')
out_dbscan = widgets.HTML()

def dbscan_svg(rows):
    width,height,left,right,top,bottom=760,360,90,735,35,270
    if not rows: return '<svg xmlns="http://www.w3.org/2000/svg" width="760" height="360"><text x="24" y="50">No stored DBSCAN configuration matches these controls.</text></svg>'
    maximum=max(1,max(max(int(float(r['clusters'])),int(float(r['noise']))) for r in rows))
    slot=(right-left)/len(rows); bar=slot*.30; pieces=[]
    for i,row in enumerate(rows):
        center=left+(i+.5)*slot
        for j,key,color,label in [(0,'clusters','#277da1','clusters'),(1,'noise','#f8961e','noise')]:
            value=int(float(row[key])); h=value*(bottom-top)/maximum; x=center+(j-.5)*bar
            pieces.append(f'<rect x="{x:.1f}" y="{bottom-h:.1f}" width="{bar*.82:.1f}" height="{h:.1f}" fill="{color}"><title>{label}: {value}</title></rect>')
        pieces.append(f'<text x="{center:.1f}" y="{bottom+22}" text-anchor="middle">eps={float(row["eps"]):.2g}</text>')
    legend='<rect x="500" y="18" width="12" height="12" fill="#277da1"/><text x="518" y="29">Clusters</text><rect x="590" y="18" width="12" height="12" fill="#f8961e"/><text x="608" y="29">Noise points</text>'
    return f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}" role="img"><title>DBSCAN clusters and noise by eps</title><desc>{len(rows)} stored parameter configurations for min_samples={int(rows[0]["min_samples"])}.</desc><rect width="100%" height="100%" fill="white"/><line x1="{left}" y1="{bottom}" x2="{right}" y2="{bottom}" stroke="#475569"/><line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#475569"/>{legend}{"".join(pieces)}<text x="{(left+right)/2}" y="{bottom+55}" text-anchor="middle">eps · Euclidean distance threshold</text><text x="20" y="155" transform="rotate(-90 20 155)" text-anchor="middle">Participants</text></svg>'

def render_dbscan(change=None):
    rows = [row for row in dbscan_scores if int(row['min_samples']) == dbscan_min_samples.value and (dbscan_eps.value == 'All eps' or float(row['eps']) == float(dbscan_eps.value))]
    rows = sorted(rows, key=lambda row: float(row['eps']))
    panel_state['dbscan_rows'] = rows
    panel_state['dbscan_svg'] = dbscan_svg(rows)
    kmeans = kmeans_evidence['all_days']
    summary = f"<p>Stored configurations shown: {len(rows)}. K-Means reference: K={kmeans['selected_k']} on 35 participants; DBSCAN grid min_samples={dbscan_min_samples.value}. No model refit.</p>"
    out_dbscan.value = panel_state['dbscan_svg'] + summary + table_html(rows, ['eps','min_samples','clusters','noise','noise_percent','silhouette','davies_bouldin'], 15)

dbscan_min_samples.observe(render_dbscan, names='value')
dbscan_eps.observe(render_dbscan, names='value')
render_dbscan()

## 6.5 Association-rule explorer
Adjust minimum support, confidence, and lift; optionally restrict to rules meeting all original eligibility thresholds and search antecedent/consequent items. The table is filtered from the 860 stored rules. No widget change reruns Apriori. Support is the joint share of participant-day transactions; confidence is conditional co-occurrence; lift compares it with the consequent base rate.

In [6]:
min_support = widgets.FloatSlider(value=.05, min=0, max=.25, step=.01, description='Min support', readout_format='.2f')
min_confidence = widgets.FloatSlider(value=.60, min=0, max=1, step=.05, description='Min confidence', readout_format='.2f')
min_lift = widgets.FloatSlider(value=1.10, min=0, max=3, step=.05, description='Min lift', readout_format='.2f')
eligible_only = widgets.Checkbox(value=True, description='Eligible only')
antecedent_search = widgets.Text(value='', description='Antecedent')
consequent_search = widgets.Text(value='', description='Consequent')
out_rules = widgets.HTML()

def render_rules(change=None):
    a_search=antecedent_search.value.casefold().strip(); c_search=consequent_search.value.casefold().strip()
    selected=[]
    for row in rules:
        if number(row,'support') < min_support.value or number(row,'confidence') < min_confidence.value or number(row,'lift') < min_lift.value: continue
        if eligible_only.value and not bool_value(row['threshold_eligible']): continue
        if a_search and a_search not in row['antecedent_items'].casefold(): continue
        if c_search and c_search not in row['consequent_items'].casefold(): continue
        selected.append(row)
    selected.sort(key=lambda row:(-number(row,'lift'),-number(row,'confidence')))
    panel_state['rule_rows']=selected
    summary = f'<p>{len(selected)} of {len(rules)} stored rules match these display filters; Apriori was not rerun.</p>'
    out_rules.value = summary + table_html(selected, ['antecedent_items','consequent_items','support','confidence','lift','distinct_participants','threshold_eligible'], 25)

for control in [min_support,min_confidence,min_lift,eligible_only,antecedent_search,consequent_search]: control.observe(render_rules, names='value')
render_rules()

## 6.6 Anomaly-candidate review
The registry contains 457 participant-day records, of which exactly 217 have `any_candidate=True`. “All flagged candidates” uses that union field; “All records” displays the full registry. Individual detector options use `candidate_methods`. Detector counts overlap and are never summed to estimate the union.

In [7]:
candidate_methods = sorted({method for row in candidates for method in json_list(row.get('candidate_methods'))})
candidate_filter = widgets.Dropdown(options=['All flagged candidates','All records'] + [f'Detector: {name}' for name in candidate_methods], value='All flagged candidates', description='Show')
out_candidates = widgets.HTML()

def render_candidates(change=None):
    mode=candidate_filter.value
    if mode == 'All records': selected=list(candidates)
    elif mode == 'All flagged candidates': selected=[row for row in candidates if bool_value(row.get('any_candidate'))]
    else:
        selected=[row for row in candidates if mode.removeprefix('Detector: ') in json_list(row.get('candidate_methods'))]
    panel_state['candidate_rows']=selected
    label='flagged candidates' if mode != 'All records' else 'records (flagged and unflagged)'
    summary = f'<p>{len(selected)} {label} shown out of {len(candidates)} registry rows. Flags are not confirmed errors.</p>'
    out_candidates.value = summary + table_html(selected, ['participant_id','date','complete_day','total_steps','total_recorded_minutes','candidate_methods','any_candidate'], 25)

candidate_filter.observe(render_candidates, names='value')
render_candidates()

## 6.7 Sleep episodes and recorded-time accounting
The time-accounting table has 23 distinct `activity_date` values. Its selector filters that table only; sleep episodes are separately filtered by their waking/calendar date. The two date sets are not treated as interchangeable. Recorded sleep rows are not verified sleep duration, and unmatched/ambiguous sleep is not imputed as zero.

In [8]:
time_dates = sorted({row['activity_date'] for row in time_accounting if row.get('activity_date')})
sleep_date = widgets.Dropdown(options=['All recorded dates'] + time_dates, description='Time date')
out_sleep = widgets.HTML()

def render_sleep(change=None):
    day=sleep_date.value
    time_rows=list(time_accounting) if day=='All recorded dates' else [row for row in time_accounting if row['activity_date']==day]
    sleep_rows=list(sleep) if day=='All recorded dates' else [row for row in sleep if row.get('waking_date')==day or row.get('calendar_date')==day]
    panel_state['time_rows']=time_rows; panel_state['sleep_rows']=sleep_rows
    scope = ('All recorded activity dates' if day == 'All recorded dates' else f'Activity date {html.escape(str(day))}')
    summary = (f'<p>Time-accounting scope: {len(time_rows)} rows for {scope}. '
               f'Sleep scope: {len(sleep_rows)} episode rows matching waking/calendar date. '
               'These counts are not interchangeable.</p>')
    activity_html = '<h4>Recorded-time accounting</h4>' + table_html(time_rows, ['activity_date','participant_days','mean_activity_minutes','mean_recorded_sleep_minutes','mean_combined_minutes','within_30_count'], 12)
    sleep_html = '<h4>Sleep episodes (separate grain)</h4>' + table_html(sleep_rows, ['id','waking_date','recorded_minute_count','primary_eligible','quality_flags'], 12)
    out_sleep.value = summary + activity_html + sleep_html

sleep_date.observe(render_sleep, names='value')
render_sleep()

## 6.8 Overview, interpretation, and next phase
The tabs assemble the interactive evidence browser. Summary values distinguish participant assignments from cluster counts and cohort sample sizes. All results remain descriptive and tied to the stored 2016 records.

In [9]:
all_cluster_count=len(panel_state.get('cluster_counts',{}))
complete_cluster_count=len({row['complete_days_cluster'] for row in clusters if row.get('complete_days_cluster','').strip()})
overview=widgets.HTML(f'''<div style="font-family:system-ui;line-height:1.6"><h2>FitInsight at a glance</h2><div style="display:flex;flex-wrap:wrap;gap:12px"><div><b>Activity records</b><br>{len(activity)} participant-days<br>35 participants</div><div><b>K-Means assignments</b><br>All-days: 35 people / {all_cluster_count} clusters<br>Complete-days: 24 people / {complete_cluster_count} clusters</div><div><b>Association rules</b><br>{len(rules)} stored rules<br>89 eligible under the original thresholds</div><div><b>Anomaly registry</b><br>217 flagged candidates<br>{len(candidates)} total records</div><div><b>Sleep/time</b><br>{len(sleep)} episodes<br>{len(time_dates)} time-accounting dates</div></div><p>Convenience sample from 2016. Coverage is not verified device wear. A cluster ID is not a health category; an association is not causal; a candidate is not a confirmed error.</p></div>''')
tabs=widgets.Tab(children=[widgets.VBox([overview]),widgets.VBox([participant,coverage,out_activity]),widgets.VBox([cluster_cohort,cluster_choice,out_clusters]),widgets.VBox([pca_cohort,pca_participant,out_pca]),widgets.VBox([dbscan_min_samples,dbscan_eps,out_dbscan]),widgets.VBox([min_support,min_confidence,min_lift,eligible_only,antecedent_search,consequent_search,out_rules]),widgets.VBox([candidate_filter,out_candidates]),widgets.VBox([sleep_date,out_sleep])])
for index,title in enumerate(['Overview','Activity','Clusters','PCA','DBSCAN','Association rules','Anomalies','Sleep / time']): tabs.set_title(index,title)
display(tabs)
print('FITINSIGHT_PROOF | P6-INTERACTIVE-NOTEBOOK | panels=8 | all_flagged=%d | all_records=%d | all_days_assignments=35 | complete_days_assignments=24 | time_dates=%d | pca_and_dbscan_refit=false' % (sum(bool_value(r['any_candidate']) for r in candidates),len(candidates),len(time_dates)))

FITINSIGHT_PROOF | P6-INTERACTIVE-NOTEBOOK | panels=8 | all_flagged=217 | all_records=457 | all_days_assignments=35 | complete_days_assignments=24 | time_dates=23 | pca_and_dbscan_refit=false


## Phase 6 conclusion and limitations
The notebook provides interactive access to validated tables and stored model summaries, while all model fitting remains in Phases 4–5. K-Means and DBSCAN outputs are exploratory, PC axes represent standardized features, association rules are not causal, anomaly flags are screening prompts, and time totals do not prove 24-hour wear or actual sleep duration. The optional web dashboard is a separate future component; Phase 7 is outside this branch.